# Deep Learning Introduction with PyTorch

Build intuition from first principles: tensors → autograd → neural networks → training loops.

**Topics:** PyTorch tensors, autograd, nn.Module, DataLoader, training loop, activation functions, batch normalization, dropout

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    print(f'PyTorch version: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
except ImportError:
    print('Install PyTorch: pip install torch torchvision')
    print('Alternatively, run in Google Colab (PyTorch pre-installed)')

np.random.seed(42)
torch.manual_seed(42)

## 1. PyTorch Tensors — The Building Block

In [ ]:
import torch

# Tensor creation
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
zeros = torch.zeros(3, 4)
rand_normal = torch.randn(2, 3)  # standard normal
from_numpy = torch.from_numpy(np.array([1.0, 2.0, 3.0]))

print('Tensor x:')
print(x)
print(f'Shape: {x.shape}, Dtype: {x.dtype}, Device: {x.device}')

# Operations (same as NumPy, but on GPU-capable tensors)
print('\nMatrix multiply:')
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = A @ B  # equivalent to torch.matmul(A, B)
print('A shape:', A.shape, 'B shape:', B.shape, '→ C shape:', C.shape)

# NumPy interop
t = torch.tensor([1.0, 2.0, 3.0])
arr = t.numpy()  # tensor → numpy (shares memory!)
t2 = torch.from_numpy(arr)  # numpy → tensor
print('\nNumPy ↔ PyTorch conversion works seamlessly')

## 2. Autograd — Automatic Differentiation

In [ ]:
import torch

# requires_grad=True: track operations for gradient computation
x = torch.tensor(2.0, requires_grad=True)

# Forward pass: compute function
# y = x^3 + 2x^2 - 5
y = x**3 + 2*x**2 - 5
print(f'y = x^3 + 2x^2 - 5 at x=2: {y.item()}')

# Backward pass: compute gradients
y.backward()

# dy/dx = 3x^2 + 4x = 3(4) + 4(2) = 20
print(f'dy/dx at x=2: {x.grad.item()}  (analytical: 3*4 + 4*2 = {3*4 + 4*2})')

# Gradient descent step on parameters
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Simple linear regression: y = wx + b
x_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_data = torch.tensor([2.0, 4.0, 6.0, 8.0])  # y = 2x

lr = 0.01
losses = []
for epoch in range(100):
    # Forward
    y_pred = w * x_data + b
    loss = ((y_pred - y_data)**2).mean()
    losses.append(loss.item())
    
    # Backward
    loss.backward()
    
    # Update (no grad tracking for this step)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

print(f'\nAfter 100 steps: w={w.item():.4f} (true: 2.0), b={b.item():.4f} (true: 0.0)')

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Manual Gradient Descent with PyTorch Autograd')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 3. Neural Network with nn.Module

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load data
data = load_breast_cancer()
X, y = data.data, data.target
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
device = torch.device('cpu')
X_tr = torch.FloatTensor(X_train).to(device)
y_tr = torch.FloatTensor(y_train).to(device)
X_te = torch.FloatTensor(X_test).to(device)
y_te = torch.FloatTensor(y_test).to(device)

# DataLoader for batching
train_dataset = TensorDataset(X_tr, y_tr)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define network
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x).squeeze(1)

model = BinaryClassifier(X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print('Model architecture:')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
import torch

# Training loop
n_epochs = 50
train_losses, val_accuracies = [], []

for epoch in range(n_epochs):
    # Training
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_pred = model(X_te)
        val_acc = ((val_pred > 0.5).float() == y_te).float().mean().item()
    
    train_losses.append(epoch_loss / len(train_loader))
    val_accuracies.append(val_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: loss={train_losses[-1]:.4f}, val_acc={val_acc:.4f}')

print(f'\nFinal test accuracy: {val_accuracies[-1]:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, 'b-'); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
ax2.plot(val_accuracies, 'r-'); ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()